In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("shubhamchandra235/imdb-and-tmdb-movie-metadata-big-dataset-1m")

print("Path to dataset files:", path)

100%|██████████| 397M/397M [00:02<00:00, 152MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/shubhamchandra235/imdb-and-tmdb-movie-metadata-big-dataset-1m/versions/1


In [ ]:
# ===============================
# PREPROCESSING + CLASS BALANCING
# ===============================

import pandas as pd
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.feature_extraction.text import TfidfVectorizer

df = pd.read_csv("IMDB TMDB Movie Metadata Big Dataset (1M).csv",
                 engine='python', on_bad_lines='skip')
print("Dataset loaded:", df.shape)
def get_final_rating(row):

    if pd.notna(row['IMDB_Rating']):
        return row['IMDB_Rating']

    elif pd.notna(row['vote_average']):
        return row['vote_average']

    elif pd.notna(row['AverageRating']):
        return row['AverageRating']

    elif pd.notna(row['Meta_score']):
        return row['Meta_score'] / 10

    else:
        return None


df['final_rating'] = df.apply(get_final_rating, axis=1)

df = df[df['final_rating'].notna()]

print("Final rating column created!")
print(df[['IMDB_Rating','vote_average','final_rating']].head())

df = df[['title', 'overview', 'genres_list', 'Cast_list', 'Director',
         'Music_Composer', 'original_language', 'final_rating']]




for col in ['overview', 'genres_list', 'Cast_list', 'Director', 'Music_Composer', 'original_language']:
    df[col] = df[col].fillna("").str.strip()


df = df[df['overview'].str.len() > 20]
languages_to_keep = ['en', 'hi', 'ta']  # Example: English, Hindi, Tamil
df = df[df['original_language'].isin(languages_to_keep)]
print("Filtered by language:", df.shape)

df['tags'] = (df['overview'] + " " +
              df['genres_list'] + " " +
              df['Cast_list'] + " " +
              df['Director'] + " " +
              df['Music_Composer'])


def rating_category(rating):
    if rating >= 8:
        return "Excellent"
    elif rating >= 7:
        return "Good"
    else:
        return "Average"

df['rating_category'] = df['final_rating'].apply(rating_category)
print("Rating categories distribution:\n", df['rating_category'].value_counts())


X = df['tags']
y = df['rating_category']

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_train = tfidf.fit_transform(X_train_text)
X_test = tfidf.transform(X_test_text)
print("TF-IDF Vectorization completed. Shape:", X_train.shape)


smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print("After balancing:")
print(pd.Series(y_train_res).value_counts())

Dataset loaded: (46817, 42)
Final rating column created!
   IMDB_Rating  vote_average  final_rating
0          8.8         8.364           8.8
1          8.6         8.417           8.6
2          9.0         8.512           9.0
3          7.8         7.573           7.8
4          8.0         7.710           8.0
Filtered by language: (29484, 8)
Rating categories distribution:
 rating_category
Average      23132
Good          5514
Excellent      838
Name: count, dtype: int64
TF-IDF Vectorization completed. Shape: (23587, 5000)
After balancing:
rating_category
Average      18506
Good         18506
Excellent    18506
Name: count, dtype: int64


In [ ]:
# ===============================
# COSINE SIMILARITY BASED RECOMMENDATION
# ===============================

from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# -------------------------------
# TF-IDF WITH N-GRAMS
# -------------------------------

tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words='english',
    ngram_range=(1,2)   # Unigram + Bigram
)

tfidf_matrix = tfidf.fit_transform(df['tags'])

print("TF-IDF with n-grams completed!")
print("Shape:", tfidf_matrix.shape)

# -------------------------------
# Cosine Similarity
# -------------------------------
similarity = cosine_similarity(tfidf_matrix)
print("Cosine similarity matrix shape:", similarity.shape)



TF-IDF with n-grams completed!
Shape: (29484, 5000)


In [ ]:
# -------------------------------
# Recommendation Function
# -------------------------------
def recommend(movie_name, top_n=5):
    movie_name = movie_name.lower()
    indices = df[df['title'].str.lower() == movie_name].index

    if len(indices) == 0:
        print("Movie not found!")
        return

    idx = indices[0]
    distances = similarity[idx]


    movies_list = sorted(list(enumerate(distances)), key=lambda x: x[1], reverse=True)[1:top_n+1]

    print(f"\nTop {top_n} movies similar to '{df.iloc[idx]['title']}':\n")
    for i, (movie_idx, score) in enumerate(movies_list, 1):
        print(f"{i}. {df.iloc[movie_idx]['title']} (Similarity: {score:.3f})")



In [ ]:
# -------------------------------
# Test Recommendation
# -------------------------------
recommend("Inception")


Top 5 movies similar to 'Inception':

1. Taken (Similarity: 0.327)
2. Interstellar (Similarity: 0.269)
3. Harry Potter and the Half-Blood Prince (Similarity: 0.258)
4. The Amazing Spider-Man (Similarity: 0.239)
5. Robin Hood (Similarity: 0.162)


In [ ]:
# ===============================
# USER PREFERENCE RECOMMENDATION
# ===============================

def recommend_by_preferences(
        genre=None,
        actor=None,
        director=None,
        language=None,
        min_rating=None):

    results = df.copy()

    if genre:
        results = results[
            results['genres_list'].str.contains(genre, case=False)
        ]

    if actor:
        results = results[
            results['Cast_list'].str.contains(actor, case=False)
        ]

    if director:
        results = results[
            results['Director'].str.contains(director, case=False)
        ]

    if language:
        results = results[
            results['original_language'].str.contains(language, case=False)
        ]

    if min_rating:
        results = results[
            results['final_rating'] >= min_rating
        ]

    print("\nRecommended Movies:\n")

    for movie in results['title'].head(5):
        print(movie)

In [ ]:
recommend_by_preferences(
    genre="Action",
    min_rating=8,
    language="en"
)


Recommended Movies:

Inception
The Dark Knight
The Avengers
Deadpool
Avengers: Infinity War


In [ ]:
def movie_details(movie_name):
    movie = df[df['title'].str.lower() == movie_name.lower()]
    if movie.empty:
        print("Movie not found")
        return
    movie = movie.iloc[0]
    print("\nMovie Details\n")
    print("Title:", movie['title'])
    print("Genre:", movie['genres_list'])
    print("Director:", movie['Director'])
    print("Actors:", movie['Cast_list'])
    print("Language:", movie['original_language'])
    print("IMDB Rating:", movie['final_rating'])
movie_details("Inception")


Movie Details

Title: Inception
Genre: ['Action', 'Science Fiction', 'Adventure']
Director: Christopher Nolan
Actors: ['Tim Kelleher', 'Silvie Laguna', 'Natasha Beaumont', 'Kraig Thornber', 'Jack Murray', 'Adam Cole', 'Claire Geare', 'Marion Cotillard', 'Magnus Nolan', 'Tai-Li Lee', 'Shannon Welles', 'Taylor Geare', 'Tom Berenger', 'Coralie Dedykere', 'Carl Gilliard', 'Miranda Nolan', 'Earl Cameron', 'Yuji Okumoto', 'Helena Cullinan', 'Nicolas Clerc', 'Andrew Pleavin', 'Alex Lombard', 'Mark Fleischmann', 'Michael Gaston', 'Marc Raducci', 'Jack Gilroy', 'Nicole Pulliam', 'Shelley Lang', 'Lukas Haas', 'Russ Fega', 'Felix Scott', 'Ryan Hayward', 'Cillian Murphy', 'Dileep Rao', 'Jill Maddrell', 'Jean-Michel Dagory', 'Jason Tendell', 'Virgile Bramly', 'Tom Hardy', 'Tohoru Masamune', 'Michael Caine', 'Talulah Riley', 'Angela Nathenson', 'Lisa Reynolds', 'Peter Basham', 'Daniel Girondeaud', 'Johnathan Geare', 'Pete Postlethwaite']
Language: en
IMDB Rating: 8.8


In [ ]:
# ======================================
# MOOD BASED RECOMMENDATION
# ======================================

def mood_recommendation(mood):

    mood_map = {
        "happy": "Comedy",
        "sad": "Drama",
        "excited": "Action",
        "romantic": "Romance",
        "thriller": "Thriller"
    }

    if mood not in mood_map:
        print("Mood not supported")
        return

    genre = mood_map[mood]

    print("\nUser Mood:", mood)
    print("Recommended Genre:", genre)

    results = df[
        df['genres_list'].str.contains(genre, case=False)
    ]

    results = results.sort_values(
        by='final_rating',
        ascending=False
    )

    print("\nRecommended Movies:\n")

    for movie in results['title'].head(5):
        print(movie)
mood_recommendation("excited")


User Mood: excited
Recommended Genre: Action

Recommended Movies:

What's New Scooby-Doo? Vol. 4: Merry Scary Holiday
The Dark Knight
The Lord of the Rings: The Return of the King
The Lord of the Rings: The Fellowship of the Ring
Inception


In [ ]:
!pip install mlxtend

In [ ]:
# ===============================
# FP-GROWTH + ASSOCIATION RULES
# ===============================

import ast
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules

print("Starting FP-Growth...\n")


#df_fp = df.sample(2000, random_state=42)   # reduce size to avoid crash

# -------------------------------
# 2Select required columns
# -------------------------------
data = df[['genres_list', 'Cast_list', 'Director', 'original_language']]

# -------------------------------
# 3Create Transactions
# -------------------------------
transactions = []

for _, row in data.iterrows():

    items = []

    # Genres
    if row['genres_list']:
        try:
            genres = ast.literal_eval(row['genres_list'])
            items.extend([g.strip() for g in genres])
        except:
            pass

    # Actors
    if row['Cast_list']:
        try:
            actors = ast.literal_eval(row['Cast_list'])
            items.extend([a.strip() for a in actors[:3]])  # limit actors
        except:
            pass

    # Director
    if row['Director']:
        items.append(row['Director'].strip())

    # Language
    if row['original_language']:
        items.append(row['original_language'].strip())

    # Remove duplicates
    items = list(set(items))

    if len(items) > 0:
        transactions.append(items)

print("Sample Transaction:\n", transactions[0])
print("Total Transactions:", len(transactions), "\n")

# -------------------------------
# Convert to One-Hot Encoding
# -------------------------------
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)

transaction_df = pd.DataFrame(te_array, columns=te.columns_)

print("Transaction DataFrame Shape:", transaction_df.shape)

# -------------------------------
# Apply FP-Growth
# -------------------------------
frequent_itemsets = fpgrowth(
    transaction_df,
    min_support=0.05,   # increase to reduce load
    use_colnames=True
)

print("\nFrequent Itemsets:\n")
print(frequent_itemsets.head())

# -------------------------------
# Association Rules
# -------------------------------
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.6
)

# Sort by lift
rules = rules.sort_values(by="lift", ascending=False)

print("\nTop Association Rules:\n")
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))

Starting FP-Growth...



/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Sample Transaction:
 ['Adventure', 'Natasha Beaumont', 'Christopher Nolan', 'Action', 'Tim Kelleher', 'Silvie Laguna', 'Science Fiction', 'en']
Total Transactions: 21887 



/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Transaction DataFrame Shape: (21887, 50054)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
# ===============================
# PREPARE RULES FOR RECOMMENDATION
# ===============================

# Convert frozenset → list for easy use
rules['antecedents'] = rules['antecedents'].apply(lambda x: list(x))
rules['consequents'] = rules['consequents'].apply(lambda x: list(x))

print("Rules ready for recommendation!")

Rules ready for recommendation!


In [ ]:
# ===============================
# FP-GROWTH RECOMMENDATION FUNCTION
# ===============================

def fp_recommend(user_inputs):

    print("\nUser Input:", user_inputs)

    matched_rules = []

    # Find matching rules
    for _, row in rules.iterrows():

        if set(row['antecedents']).issubset(set(user_inputs)):
            matched_rules.append(row)

    if len(matched_rules) == 0:
        print("No matching patterns found!")
        return

    # Sort by lift (strongest rule first)
    matched_rules = sorted(matched_rules, key=lambda x: x['lift'], reverse=True)

    print("\nTop Matching Rule:")
    print("IF", matched_rules[0]['antecedents'], "THEN", matched_rules[0]['consequents'])

    # Get recommended features
    recommended_features = matched_rules[0]['consequents']

    # -------------------------------
    # Recommend Movies based on features
    # -------------------------------
    results = df.copy()

    for feature in recommended_features:
        results = results[
            results['genres_list'].str.contains(feature, case=False) |
            results['Cast_list'].str.contains(feature, case=False) |
            results['Director'].str.contains(feature, case=False) |
            results['original_language'].str.contains(feature, case=False)
        ]

    print("\nRecommended Movies:\n")

    for movie in results['title'].head(5):
        print(movie)

In [ ]:
fp_recommend(['Comedy', 'Family'])


User Input: ['Comedy', 'Family']

Top Matching Rule:
IF ['Family'] THEN ['en', 'Comedy']

Recommended Movies:

Deadpool
Forrest Gump
The Wolf of Wall Street
Inside Out
Up


In [ ]:
# ============================================
# CLASSIFICATION: NAIVE BAYES + LOGISTIC REGRESSION
# ============================================

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

print("\nStarting Classification Models...\n")

# -------------------------------
# Data (Already Balanced)
# -------------------------------
# X_train_res, y_train_res → from SMOTE
# X_test, y_test → from earlier split

# ===============================
# NAIVE BAYES MODEL
# ===============================

nb_model = MultinomialNB()
nb_model.fit(X_train_res, y_train_res)

# Prediction
y_pred_nb = nb_model.predict(X_test)

# Evaluation
print("🔹 Naive Bayes Results:\n")
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_nb))


# ===============================
# LOGISTIC REGRESSION MODEL
# ===============================

lr_model = LogisticRegression(
    max_iter=2000,
    C=2,              # stronger learning
    solver='lbfgs'
)
lr_model.fit(X_train_res, y_train_res)

# Prediction
y_pred_lr = lr_model.predict(X_test)

# Evaluation
print("\n🔹 Logistic Regression Results:\n")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_lr))


# ===============================
# PREDICTION FUNCTION
# ===============================

def predict_movie_category(movie_name, model_type='lr'):

    movie_index = df[df['title'].str.lower() == movie_name.lower()].index

    if len(movie_index) == 0:
        print("Movie not found!")
        return

    idx = movie_index[0]

    # Convert to TF-IDF vector
    movie_vector = tfidf.transform([df.iloc[idx]['tags']])

    # Choose model
    model = lr_model if model_type == 'lr' else nb_model

    prediction = model.predict(movie_vector)

    print("\n🎬 Movie:", movie_name)
    print("Predicted Category:", prediction[0])


# ===============================
# TEST PREDICTION
# ===============================

predict_movie_category("Inception", model_type='nb')
predict_movie_category("Inception", model_type='lr')


Starting Classification Models...

🔹 Naive Bayes Results:

Accuracy: 0.42111801242236024

Classification Report:

              precision    recall  f1-score   support

     Average       0.57      0.47      0.51       440
   Excellent       0.18      0.30      0.22        79
        Good       0.35      0.38      0.37       286

    accuracy                           0.42       805
   macro avg       0.37      0.38      0.37       805
weighted avg       0.46      0.42      0.43       805


🔹 Logistic Regression Results:

Accuracy: 0.5167701863354037

Classification Report:

              precision    recall  f1-score   support

     Average       0.63      0.62      0.62       440
   Excellent       0.27      0.22      0.24        79
        Good       0.41      0.44      0.42       286

    accuracy                           0.52       805
   macro avg       0.44      0.43      0.43       805
weighted avg       0.52      0.52      0.52       805


🎬 Movie: Inception
Predicted Catego

In [ ]:
import joblib

print("Exporting models and data...")

# 1. Save the cleaned DataFrame (for movie titles and details)
joblib.dump(df, 'movie_database.pkl')

# 2. Save the Cosine Similarity matrix (for content-based recommendations)
joblib.dump(similarity, 'cosine_sim.pkl')

# 3. Save the FP-Growth association rules (for basket recommendations)
joblib.dump(rules, 'fp_growth_rules.pkl')

# 4. Save the TF-IDF Vectorizer (needed to transform new inputs for the ML model)
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')

# 5. Save the trained Logistic Regression model (for rating prediction)
joblib.dump(lr_model, 'logistic_model.pkl')

print("All 5 files successfully saved as .pkl!")

Exporting models and data...
✅ All 5 files successfully saved as .pkl!
